In [2]:
import geopandas as gpd
import pandas as pd

# Roads layer with COGID but missing lanes
roads_gdf = gpd.read_file("../TransitModeShare2024.gpkg")

# CSV with COGID + lane count
lanes_df = pd.read_csv("AWDT.csv")

lanes_df.shape

(3044, 20)

In [3]:
roads_gdf.shape

(3050, 59)

In [4]:
lanes_df.dtypes

COG ID                                 int64
Length (Mi)                          float64
Functional Class                      object
Route                                 object
Location                              object
Speed                                  int64
Lanes                                  int64
Average Weekday Traffic 2018           int64
Average Weekday Traffic 2019           int64
Average Weekday Traffic 2020           int64
Average Annual Daily Traffic 2018      int64
Average Annual Daily Traffic 2019      int64
Average Annual Daily Traffic 2020      int64
AWDT21                                 int64
AWDT22                                 int64
ADT21                                  int64
ADT22                                  int64
VMT                                    int64
VHT                                    int64
VHD                                    int64
dtype: object

In [5]:
roads_gdf.dtypes

OBJECTID              int64
COGID               float64
RTE                  object
LOCAT                object
LENGTH              float64
MILES               float64
ADT22               float64
AWDT22              float64
Shape_Length        float64
ADT23               float64
AWDT23              float64
RT1_NB                int32
RT1_SB                int32
RT2_NB                int32
RT2_SB                int32
RT5_EB                int32
RT5_WB                int32
RT8_EB                int32
RT8_WB                int32
RT10_NB               int32
RT10_SB               int32
RT11_EB               int32
RT11_WB               int32
RT16_EB               int32
RT16_WB               int32
RT31_NB               int32
RT31_SB               int32
RT36                  int32
RT50_NB               int32
RT50_SB               int32
RT51_NB               int32
RT51_SB               int32
RT53_NB               int32
RT53_SB               int32
RT54_EB               int32
RT54_WB             

In [8]:
import numpy as np
import pandas as pd
import geopandas as gpd

# Read
roads_gdf = gpd.read_file("../TransitModeShare2024.gpkg")          # has COGID as int64 already (say)
lanes_df  = pd.read_csv("AWDT.csv")        # has COG ID as float64

# Standardize the column name
lanes_df = lanes_df.rename(columns={"COG ID": "COGID"})

# If float keys are really integers (e.g., 123.0), make them ints
# First, verify there are no fractional values:
bad = lanes_df.loc[~(lanes_df["COGID"] % 1 == 0), "COGID"]
assert bad.empty, f"Found non-integer COGIDs: {bad.unique()[:10]}"

# Convert to pandas' nullable integer (handles NaN safely)
lanes_df["COGID"] = lanes_df["COGID"].astype("Int64")
roads_gdf["COGID"] = roads_gdf["COGID"].astype("Int64")
lanes_df.dtypes

COGID                                  Int64
Length (Mi)                          float64
Functional Class                      object
Route                                 object
Location                              object
Speed                                  int64
Lanes                                  int64
Average Weekday Traffic 2018           int64
Average Weekday Traffic 2019           int64
Average Weekday Traffic 2020           int64
Average Annual Daily Traffic 2018      int64
Average Annual Daily Traffic 2019      int64
Average Annual Daily Traffic 2020      int64
AWDT21                                 int64
AWDT22                                 int64
ADT21                                  int64
ADT22                                  int64
VMT                                    int64
VHT                                    int64
VHD                                    int64
dtype: object

In [9]:
roads_gdf.dtypes

OBJECTID              int64
COGID                 Int64
RTE                  object
LOCAT                object
LENGTH              float64
MILES               float64
ADT22               float64
AWDT22              float64
Shape_Length        float64
ADT23               float64
AWDT23              float64
RT1_NB                int32
RT1_SB                int32
RT2_NB                int32
RT2_SB                int32
RT5_EB                int32
RT5_WB                int32
RT8_EB                int32
RT8_WB                int32
RT10_NB               int32
RT10_SB               int32
RT11_EB               int32
RT11_WB               int32
RT16_EB               int32
RT16_WB               int32
RT31_NB               int32
RT31_SB               int32
RT36                  int32
RT50_NB               int32
RT50_SB               int32
RT51_NB               int32
RT51_SB               int32
RT53_NB               int32
RT53_SB               int32
RT54_EB               int32
RT54_WB             

In [10]:
# If duplicates exist, decide how to resolve them (keep first, or aggregate)
lanes_df = lanes_df.sort_values("COGID").drop_duplicates("COGID", keep="first")
lanes_df.shape

(3044, 20)

In [11]:
cols_to_add = ["COGID", "Lanes"]   # keep it tight so you don't accidentally overwrite fields
merged = roads_gdf.merge(lanes_df[cols_to_add], on="COGID", how="left", validate="m:1")
merged.head()

,OBJECTID,COGID,RTE,LOCAT,LENGTH,MILES,ADT22,AWDT22,Shape_Length,ADT23,...,RT766_EB,RT766_WB,RT777_EB,RT777_WB,AWDT23*1.2,Total_Users,TransitUsersTTL,TrnstModeShare,geometry,Lanes
0,1,<NA>,None,None,NaN,NaN,NaN,NaN,954.968271,NaN,...,0,0,0,0,NaN,NaN,0,NaN,"MULTILINESTRING ((1537901.273 1518885.412, 153...",NaN
1,2,<NA>,None,NORTH OF MEADE - SOUTH OF CONDERSHIRE,NaN,NaN,NaN,NaN,841.849864,NaN,...,0,0,0,0,NaN,NaN,0,NaN,"MULTILINESTRING ((1497526.515 1463666.515, 149...",NaN
2,3,10003,UNSER BLVD.,NORTH OF NORTHERN - SOUTH OF 15TH AVE.,2106.069,0.399,22160.0,23860.0,2106.068785,21852.0,...,0,0,0,0,28235.0,28235.0,0,0.0,"MULTILINESTRING ((1507059 1555528.75, 1507049....",4.0
3,4,10004,N.M. 528,NORTH OF CORRALES RD. (NM 448) - .118 MILES N...,648.945,0.123,25188.0,27222.0,669.271899,25134.0,...,0,0,0,0,32213.0,32213.0,0,0.0,"MULTILINESTRING ((1530760.337 1555735.639, 153...",4.0
4,5,10005,GRANDE VISTA,NORTH OF CORRALES RD. - SOUTH OF SANDIA VISTA,1070.855,0.203,2087.0,2333.0,1033.030125,2058.0,...,0,0,0,0,2761.0,2761.0,0,0.0,"MULTILINESTRING ((1531922.807 1555068.596, 153...",2.0


In [12]:
merged.dtypes

OBJECTID              int64
COGID                 Int64
RTE                  object
LOCAT                object
LENGTH              float64
MILES               float64
ADT22               float64
AWDT22              float64
Shape_Length        float64
ADT23               float64
AWDT23              float64
RT1_NB                int32
RT1_SB                int32
RT2_NB                int32
RT2_SB                int32
RT5_EB                int32
RT5_WB                int32
RT8_EB                int32
RT8_WB                int32
RT10_NB               int32
RT10_SB               int32
RT11_EB               int32
RT11_WB               int32
RT16_EB               int32
RT16_WB               int32
RT31_NB               int32
RT31_SB               int32
RT36                  int32
RT50_NB               int32
RT50_SB               int32
RT51_NB               int32
RT51_SB               int32
RT53_NB               int32
RT53_SB               int32
RT54_EB               int32
RT54_WB             

In [13]:
import pandas as pd
import numpy as np

# 1) sanity check: are there any non-whole values?
non_whole = merged["Lanes"].notna() & (merged["Lanes"] % 1 != 0)
print(f"Non-whole lane values: {merged.loc[non_whole, 'Lanes'].unique()}")

# 2) if that prints nothing (or you’re OK rounding), convert to nullable int
merged["Lanes"] = pd.to_numeric(merged["Lanes"], errors="coerce").round().astype("Int64")


Non-whole lane values: []


In [14]:
merged.head()

,OBJECTID,COGID,RTE,LOCAT,LENGTH,MILES,ADT22,AWDT22,Shape_Length,ADT23,...,RT766_EB,RT766_WB,RT777_EB,RT777_WB,AWDT23*1.2,Total_Users,TransitUsersTTL,TrnstModeShare,geometry,Lanes
0,1,<NA>,None,None,NaN,NaN,NaN,NaN,954.968271,NaN,...,0,0,0,0,NaN,NaN,0,NaN,"MULTILINESTRING ((1537901.273 1518885.412, 153...",<NA>
1,2,<NA>,None,NORTH OF MEADE - SOUTH OF CONDERSHIRE,NaN,NaN,NaN,NaN,841.849864,NaN,...,0,0,0,0,NaN,NaN,0,NaN,"MULTILINESTRING ((1497526.515 1463666.515, 149...",<NA>
2,3,10003,UNSER BLVD.,NORTH OF NORTHERN - SOUTH OF 15TH AVE.,2106.069,0.399,22160.0,23860.0,2106.068785,21852.0,...,0,0,0,0,28235.0,28235.0,0,0.0,"MULTILINESTRING ((1507059 1555528.75, 1507049....",4
3,4,10004,N.M. 528,NORTH OF CORRALES RD. (NM 448) - .118 MILES N...,648.945,0.123,25188.0,27222.0,669.271899,25134.0,...,0,0,0,0,32213.0,32213.0,0,0.0,"MULTILINESTRING ((1530760.337 1555735.639, 153...",4
4,5,10005,GRANDE VISTA,NORTH OF CORRALES RD. - SOUTH OF SANDIA VISTA,1070.855,0.203,2087.0,2333.0,1033.030125,2058.0,...,0,0,0,0,2761.0,2761.0,0,0.0,"MULTILINESTRING ((1531922.807 1555068.596, 153...",2


In [15]:
merged.to_file("../TransitModeShare2024.gpkg", layer="major_roads_with_lanes", driver="GPKG")
